# Attach worksite clusters and status to the analysis frame

Joins the cross-year worksite bridge (`pipeline/bridge.py` → `pipeline/flag_worksite_status.py`) onto the analysis frame on `(Year, worksite_id)` and writes `data/to_use/analysis_frame_worksite.csv`, which is **Frame A** for every model.

In [ ]:
# Run from anywhere inside the repo: find the root, make it the working
# directory, and put it on sys.path so `pipeline/`, `descriptives/` and
# `paths.py` import cleanly. Data paths below are relative to the root.
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'paths.py').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd

In [3]:
worksite_bridge = pd.read_csv(
    "data/to_use/worksite_bridge_status.csv"
)

analysis_frame = pd.read_csv(
    "data/to_use/analysis_frame.csv"
)

# Keep only the fields we want to add
worksite_lookup = (
    worksite_bridge[
        [
            "Year",
            "worksite_id", 
            "worksite_name_canonical",
            "worksite_cluster_id", 
            "worksite_status", 
            "is_no_show"
        ]
    ]
    .drop_duplicates(["Year", "worksite_id"])
)

# Defensive check: one worksite per Year + ID
assert not worksite_lookup.duplicated(
    ["Year", "worksite_id"]
).any()


# Add worksite information to analysis_frame
analysis_frame = analysis_frame.merge(
    worksite_lookup,
    on=["Year", "worksite_id"],
    how="left",
    validate="many_to_one",
)

In [5]:
analysis_frame.to_csv('data/to_use/analysis_frame_worksite.csv', index=False)